# Data Cleaning

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_style("whitegrid")

## Load Data

In [67]:
df = pd.read_csv("../data/raw/training_set_DM.csv")

# Initial Cleaning and Remove of Leakage Features

In [68]:
print(df.shape)
df.head()

(1048575, 54)


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,4/4/2013 8:32,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,4/4/2013 8:32,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,4/4/2013 8:32,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,4/4/2013 8:32,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,4/4/2013 8:32,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0


In [69]:
# Leakage Features
df = df.drop(columns=[
    'gross_bookings_usd',
    'position'
])

# Missing Value Handling

In [70]:
# A - prop_location_score 2

df['prop_location_score2_missing'] = (
    df['prop_location_score2']
    .isnull()
    .astype(int)
)

df['prop_location_score2'] = (
    df['prop_location_score2']
    .fillna(-1)
)

In [71]:
#B - Visitor history features

df['visitor_hist_missing'] = (
    df['visitor_hist_starrating']
    .isnull()
    .astype(int)
)

df['visitor_hist_starrating'] = (
    df['visitor_hist_starrating']
    .fillna(0)
)

df['visitor_hist_adr_usd'] = (
    df['visitor_hist_adr_usd']
    .fillna(0)
)

| value | flag |
| ----- | ---- |
| 4.5   | 0    |
| 0     | 1    |
| 3.0   | 0    |
| 0     | 1    |

0 could mean:
- real value
- OR missing

In [72]:
df['affinity_missing'] = (
    df['srch_query_affinity_score']
    .isnull()
    .astype(int)
)

df['srch_query_affinity_score'] = (
    df['srch_query_affinity_score']
    .fillna(-999)
)

# Outlier Handling

In [93]:
# p99.9 clipping

print(df['price_usd'].max())

upper_price = df['price_usd'].quantile(0.99)

df['price_usd'] = df['price_usd'].clip(
    upper=upper_price
)

print(df['price_usd'].max())

2198.3017822360925
601.58


# Feature Engineering

In [74]:
# mean price per search

df['price_mean_search'] = (
    df.groupby('srch_id')['price_usd']
    .transform('mean')
)

In [75]:
# relative price

df['price_relative'] = (
    df['price_usd'] /
    df['price_mean_search']
)

In [76]:
# log price
df['price_log'] = np.log1p(df['price_usd'])

In [77]:
# cheapest ranking

df['price_rank'] = (
    df.groupby('srch_id')['price_usd']
    .rank(method='dense')
)

In [78]:
# overall rating

df['rating_review_product'] = (
    df['prop_starrating'] *
    df['prop_review_score']
)

In [79]:
# high rating flag

df['high_rating'] = (
    df['prop_starrating'] >= 4
).astype(int)

In [80]:
# promotion flag variable

In [81]:
# type of stay

df['is_family'] = (
    df['srch_children_count'] > 0
).astype(int)

df['is_long_stay'] = (
    df['srch_length_of_stay'] > 3
).astype(int)

# Competitors

In [82]:
comp_rate_cols = [
    col for col in df.columns
    if col.startswith('comp')
    and col.endswith('_rate')
]

In [83]:
df['num_competitors_available'] = (
    df[comp_rate_cols]
    .notnull()
    .sum(axis=1)
)

In [84]:
df['expedia_vs_competitors'] = (
    df[comp_rate_cols]
    .max(axis=1)
)

In [85]:
df.isnull().sum().sort_values(ascending=False).head(20)

comp1_rate_percent_diff    1029342
comp6_rate_percent_diff    1028475
comp1_rate                 1024260
comp1_inv                  1022207
comp4_rate_percent_diff    1020768
comp7_rate_percent_diff    1019514
comp6_rate                  997730
comp6_inv                   993328
comp4_rate                  982781
comp7_rate                  982217
comp4_inv                   975276
comp7_inv                   973740
comp3_rate_percent_diff     948360
comp2_rate_percent_diff     931121
comp8_rate_percent_diff     919122
comp5_rate_percent_diff     871629
comp3_rate                  723812
comp3_inv                   698939
comp8_rate                  643233
comp8_inv                   628074
dtype: int64

In [86]:
df.to_csv(
    "../data/processed/train_prepared.csv",
    index=False
)